## GPT method

Retrieving DB

In [2]:
# Install kagglehub
!pip install kagglehub --quiet

import kagglehub
import pandas as pd
import os
from google.colab import files

# Download dataset
path = kagglehub.dataset_download("andrezaza/clapper-massive-rotten-tomatoes-movies-and-reviews")
print("Chemin du dataset :", path)

# Load and read CSV file
file_path = os.path.join(path, "rotten_tomatoes_movie_reviews.csv")
data = pd.read_csv(file_path)

# Only keep useful columns
filtered_df = data[['reviewText', 'scoreSentiment']].dropna()

# Folder creation
output_dir = "movie_reviews_dataset"
os.makedirs(output_dir, exist_ok=True)

# Save CSV file
output_path = os.path.join(output_dir, "all_reviews.csv")
filtered_df.to_csv(output_path, index=False)

print(f"Fichier enregistré à : {output_path}")

# Download file
files.download(output_path)


Chemin du dataset : /kaggle/input/clapper-massive-rotten-tomatoes-movies-and-reviews
Fichier enregistré à : movie_reviews_dataset/all_reviews.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### DATA

In [20]:
dataSet = pd.read_csv("movie_reviews_dataset/all_reviews.csv")

In [21]:
len(dataSet)

1375738

In [22]:
for index, row in dataSet.head(10).iterrows():
    print(f"Review: {row['reviewText']}")
    print(f"Sentiment: {row['scoreSentiment']}")
    print()

Review: Timed to be just long enough for most youngsters' brief attention spans -- and it's packed with plenty of interesting activity, both on land and under the water.
Sentiment: POSITIVE

Review: It doesn't matter if a movie costs 300 million or only 300 dollars; good is good and bad is bad, and Bloodmask: The Possession of Nicole Lameroux is just plain bad.
Sentiment: NEGATIVE

Review: The choreography is so precise and lifelike at points one might wonder whether the movie was rotoscoped, but no live-action reference footage was used. The quality is due to the skill of the animators and Kodama's love for professional wrestling.
Sentiment: POSITIVE

Review: The film's out-of-touch attempts at humor may find them hunting for the reason the franchise was so popular in the first place.
Sentiment: NEGATIVE

Review: Its clumsy determination is endearing and sometimes wildly entertaining
Sentiment: POSITIVE

Review: With every new minute, there's another head-scratching choice that's boun

Importations

In [32]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import wandb
from nltk.corpus import stopwords
import nltk

Downloading stop words in english

In [33]:
nltk.download("stopwords")
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Connection to GPU

In [26]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
torch.manual_seed(100)

Device: cuda


Hyperparameters

In [27]:
batch_size = 32
block_size = 512
n_embedding = 512
n_blocks = 12
n_head = 8
dropout = 0.2
max_iters = 50000
learning_rate = 3e-4
eval_every = 500
eval_iters = 200
save_every = 10000

Do not know utility...

In [28]:
model_name = "gpt-v7"
directory = "/scratch/users/glouppe/gpt"
checkpoint = f"{directory}/{model_name}.pt"

Wandb

In [29]:
wandb.init(
    project="sentiment-analysis-gpt",
    config={
        "model": model_name,
        "batch_size": batch_size,
        "block_size": block_size,
        "n_embedding": n_embedding,
        "n_blocks": n_blocks,
        "n_head": n_head,
        "dropout": dropout,
        "max_iters": max_iters,
        "learning_rate": learning_rate,
        "eval_every": eval_every,
        "eval_iters": eval_iters,
    }
)

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:


Abort: 

Dataset preparation

In [24]:
dataSet['scoreSentiment'] = dataSet['scoreSentiment'].map({'POSITIVE': 1, 'NEGATIVE': 0})

# Preprocessing step removing all ponctuations and special characters
dataSet['reviewText'] = dataSet['reviewText'].str.replace('[^\w\s]','', regex=True)

# Print 10 first scoreSentiment
print(dataSet['scoreSentiment'].head(10))

# Data conversion into lists
reviews = dataSet['reviewText'].tolist()
labels = dataSet['scoreSentiment'].tolist()

# Compute the index to split the dataset (90% train, 10% test)
n = int(0.9 * len(reviews))

# Split the dataset into training and validation sets
train_texts = reviews[:n]
train_labels = labels[:n]
test_texts = reviews[n:]
test_labels = labels[n:]

print(f"Length of train dataset : {len(train_texts)}")
print(f"Length of test dataset : {len(test_texts)}")

# Print the first 5 elements of the training and testing sets
for i in range(5):
    print(f"Text: {train_texts[i]}")
    print(f"Label: {train_labels[i]}")
    print()

for i in range(5):
    print(f"Text: {test_texts[i]}")
    print(f"Label: {test_labels[i]}")
    print()

0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
5   NaN
6   NaN
7   NaN
8   NaN
9   NaN
Name: scoreSentiment, dtype: float64
Length of train dataset : 1238164
Length of test dataset : 137574
Text: Timed to be just long enough for most youngsters brief attention spans  and its packed with plenty of interesting activity both on land and under the water
Label: nan

Text: It doesnt matter if a movie costs 300 million or only 300 dollars good is good and bad is bad and Bloodmask The Possession of Nicole Lameroux is just plain bad
Label: nan

Text: The choreography is so precise and lifelike at points one might wonder whether the movie was rotoscoped but no liveaction reference footage was used The quality is due to the skill of the animators and Kodamas love for professional wrestling
Label: nan

Text: The films outoftouch attempts at humor may find them hunting for the reason the franchise was so popular in the first place
Label: nan

Text: Its clumsy determination is endearing and sometimes wildl

Tokenization

In [34]:
# Tokenization with ByteLevelBPETokenizer
from tokenizers import ByteLevelBPETokenizer

# Tokenizer train
with open("reviews_corpus.txt", "w", encoding="utf-8") as f:
    for review in reviews:
        f.write(review + "\n")

tokenizer = ByteLevelBPETokenizer()
tokenizer.train(files=["reviews_corpus.txt"], vocab_size=30000, min_frequency=2)

# Text encoding
encode = lambda s: tokenizer.encode(s).ids
train_tokens = [encode(text) for text in train_texts]
test_tokens = [encode(text) for text in test_texts]

KeyboardInterrupt: 

Removing stop words which are the most common words in any language and does not add much information to the text

In [ ]:
dataSet['reviews'] = dataSet['reviews'].apply(lambda x: ' '.join([word for word in x if word not in (stop_words)]))
print(dataSet.head(20))


# References
https://medium.com/@lvdeep9/sentiment-analysis-on-imdb-dataset-with-gpt-3-and-gpt-3-5-b9ae8c5bc910